# Grid Finding — Orchestrator

Runs the Grid Finding pipeline: notebooks 01–06 extract features per grid cell, then merges all CSVs into a single combined dataset for ML classification (notebook 07) and heatmap visualization (notebook 08).

**Pipeline:**
```
01_grid_definition          → cell_id, zone_type (Y variable)
02_amenity_composition      → amenity_density, amenity_ratio_food_drink (OSM)
03_building_characteristics → avg_floors, avg_yearbuilt, building_count, total_bldg_area (PLUTO)
04_land_use_mix             → landuse_entropy (PLUTO)
05_tourism_intensity        → tourism_density (OSM)
06_commercial_density       → shop_density_km2, shop_type_entropy, brand_ratio (OSM)
──────────────────────────────────────────────────────────────────
07_ml_classification        → train & evaluate models, export predictions
08_heatmap_visualization    → static + interactive heatmaps
```

**Usage:** Change parameters in the cell below → Restart kernel → Run All Cells

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║           PIPELINE PARAMETERS — CHANGE HERE         ║
# ╚══════════════════════════════════════════════════════╝

# ── Borough selection ─────────────────────────────────
# Available boroughs in NYC PLUTO:
#   "MN"  →  Manhattan      (~1,810 cells at 150m)
#   "BK"  →  Brooklyn       (~4,500+ cells at 150m)
#   "QN"  →  Queens         (~6,000+ cells at 150m)
#   "BX"  →  Bronx          (~2,500+ cells at 150m)
#   "SI"  →  Staten Island  (~2,000+ cells at 150m)
#
# You can combine boroughs: ["MN", "BK"] runs both.
# Cell counts are estimates — actual count depends on
# lot density and min_lots_per_cell threshold.

BOROUGH = ["MN"]            # ← CHANGE THIS

# ── Grid parameters ──────────────────────────────────
CELL_SIZE_M = 150            # grid cell size in meters
MIN_LOTS_PER_CELL = 3        # cells with fewer lots are dropped

# ── Classification mode ──────────────────────────────
INCLUDE_OTHER = False         # False = binary (Commercial vs Residential)
                              # True  = 3-class (+ Other)

# ── PLUTO path ───────────────────────────────────────
PLUTO_PATH = "../ramy/NYC_pluto_25v4_csv/pluto_25v4.csv"

In [ ]:
# ── Write grid.json from parameters above ─────────────
import json

_BOROUGH_CODES = {"MN": 1, "BX": 2, "BK": 3, "QN": 4, "SI": 5}
_BOROUGH_NAMES = {"MN": "Manhattan", "BK": "Brooklyn", "QN": "Queens",
                  "BX": "Bronx", "SI": "Staten Island"}

# Validate borough codes
for b in BOROUGH:
    if b not in _BOROUGH_CODES:
        raise ValueError(f"Unknown borough '{b}'. Use: {list(_BOROUGH_CODES.keys())}")

config = {
    "borough_filter": BOROUGH,
    "borough_codes": _BOROUGH_CODES,
    "grid_cell_size_m": CELL_SIZE_M,
    "min_lots_per_cell": MIN_LOTS_PER_CELL,
    "pluto_path": PLUTO_PATH,
    "include_other": INCLUDE_OTHER,
    "feature_flags": {"needs_pluto": True},
    "zone_type_rules": {
        "Residential": {"landuse": "01", "threshold": 0.70},
        "Commercial": {"landuse": "02", "threshold": 0.50},
        "Industrial": {"landuse": "05", "threshold": 0.30},
        "Institutional": {"landuse": "08", "threshold": 0.30},
        "Open Space": {"landuse": ["09", "11"], "threshold": 0.30},
        "Mixed-Use": {"description": "No single category dominates"},
    },
}

with open("grid.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

borough_names = ", ".join(_BOROUGH_NAMES[b] for b in BOROUGH)
print(f"grid.json written:")
print(f"  Boroughs:       {borough_names} ({BOROUGH})")
print(f"  Cell size:      {CELL_SIZE_M}m x {CELL_SIZE_M}m")
print(f"  Min lots/cell:  {MIN_LOTS_PER_CELL}")
print(f"  Include Other:  {INCLUDE_OTHER}")
print(f"  PLUTO path:     {PLUTO_PATH}")

In [ ]:
import papermill as pm
import pandas as pd
import pathlib
import time
import os
import tempfile
from datetime import datetime

os.makedirs("csv", exist_ok=True)

RUN_ID = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
print(f"papermill {pm.__version__}")
print(f"Run ID: {RUN_ID}")

In [ ]:
# ── Pipeline configuration ────────────────────────────

PIPELINE = [
    ("01_grid_definition.ipynb",          "01 · Grid Definition (Y variable)"),
    ("02_amenity_composition.ipynb",      "02 · Amenity Composition (OSM)"),
    ("03_building_characteristics.ipynb", "03 · Building Characteristics (PLUTO)"),
    ("04_land_use_mix.ipynb",             "04 · Land Use Mix (PLUTO)"),
    ("05_tourism_intensity.ipynb",        "05 · Tourism Intensity (OSM)"),
    ("06_commercial_density.ipynb",       "06 · Commercial Density (OSM)"),
]

RUN_MODULE = {nb: True for nb, _ in PIPELINE}
KERNEL_NAME = "python3"
RUN_ML = True
RUN_HEATMAP = True

print("Pipeline:")
for nb, desc in PIPELINE:
    status = "RUN" if RUN_MODULE[nb] else "SKIP"
    print(f"  [{status}]  {desc}")
print(f"  [{'RUN' if RUN_ML else 'SKIP'}]  07 · ML Classification")
print(f"  [{'RUN' if RUN_HEATMAP else 'SKIP'}]  08 · Heatmap Visualization")

In [ ]:
# ── Run feature extraction notebooks ──────────────────

results = []

for nb_path, description in PIPELINE:
    if not RUN_MODULE.get(nb_path, True):
        print(f"  SKIP  {description}")
        results.append((nb_path, "skipped", 0))
        continue

    if not pathlib.Path(nb_path).exists():
        raise FileNotFoundError(f"Notebook not found: {nb_path}")

    print(f"\n{'='*55}")
    print(f"  {description}")
    print(f"{'='*55}")

    tmp = pathlib.Path(tempfile.mktemp(suffix=".ipynb"))
    t0 = time.time()
    try:
        pm.execute_notebook(
            nb_path, str(tmp),
            kernel_name=KERNEL_NAME,
            parameters={"GRID_CONFIG": "grid.json"},
            progress_bar=False,
        )
        elapsed = time.time() - t0
        print(f"  Status : OK  ({elapsed:.1f} s)")
        results.append((nb_path, "ok", elapsed))
    except pm.exceptions.PapermillExecutionError as e:
        elapsed = time.time() - t0
        print(f"  Status : FAILED  ({elapsed:.1f} s)")
        print(f"  Error  : {e}")
        results.append((nb_path, "failed", elapsed))
    finally:
        if tmp.exists():
            tmp.unlink()

# ── Summary ───────────────────────────────────────────
print(f"\n{'#'*55}")
print(f"  PIPELINE SUMMARY")
print(f"{'#'*55}")
for nb_path, status, elapsed in results:
    icon = {"ok": "OK", "failed": "FAIL", "skipped": "SKIP"}.get(status, "?")
    t_str = f"{elapsed:.1f} s" if elapsed else "-"
    print(f"  [{icon:4s}]  {nb_path:<42s} {t_str}")

failed = [nb for nb, s, _ in results if s == "failed"]
if failed:
    print(f"\n  WARNING: {len(failed)} notebook(s) failed: {failed}")

In [ ]:
# ── Merge all CSV outputs ─────────────────────────────

CSV_FILES = [
    "csv/01_grid_definition.csv",
    "csv/02_amenity_composition.csv",
    "csv/03_building_characteristics.csv",
    "csv/04_land_use_mix.csv",
    "csv/05_tourism_intensity.csv",
    "csv/06_commercial_density.csv",
]

# Start with grid definitions
df_combined = pd.read_csv(CSV_FILES[0], dtype={"cell_id": str})
print(f"Base: {len(df_combined)} cells from 01_grid_definition")

# Merge each subsequent CSV on cell_id
for csv_path in CSV_FILES[1:]:
    if pathlib.Path(csv_path).exists():
        df_other = pd.read_csv(csv_path, dtype={"cell_id": str})
        merge_cols = [c for c in df_other.columns if c != "cell_id"]
        df_combined = df_combined.merge(
            df_other[["cell_id"] + merge_cols],
            on="cell_id", how="left"
        )
        print(f"  OK    {csv_path} (+{len(merge_cols)} cols)")
    else:
        print(f"  WARN  {csv_path} not found")

print(f"\nCombined shape: {df_combined.shape}")

In [ ]:
# ── Save combined CSV ─────────────────────────────────

combined_path = f"csv/combined_grid_{RUN_ID}.csv"
df_combined.to_csv(combined_path, index=False, encoding="utf-8")
print(f"Saved: {combined_path}")
print(f"  {df_combined.shape[0]} cells x {df_combined.shape[1]} columns")

print(f"\nColumn completeness:")
for col in df_combined.columns:
    n = df_combined[col].notna().sum()
    pct = 100 * n / len(df_combined)
    print(f"  {col:<30s} {n:>5d}/{len(df_combined)}  ({pct:.1f}%)")

In [ ]:
# ── Run ML notebook ──────────────────────────────────

if RUN_ML:
    print(f"\n{'='*55}")
    print(f"  07 · ML Classification")
    print(f"{'='*55}")

    tmp = pathlib.Path(tempfile.mktemp(suffix=".ipynb"))
    t0 = time.time()
    try:
        pm.execute_notebook(
            "07_ml_classification.ipynb", str(tmp),
            kernel_name=KERNEL_NAME,
            parameters={"CSV_PATH": combined_path},
            progress_bar=False,
        )
        elapsed = time.time() - t0
        print(f"  Status : OK  ({elapsed:.1f} s)")
    except pm.exceptions.PapermillExecutionError as e:
        elapsed = time.time() - t0
        print(f"  Status : FAILED  ({elapsed:.1f} s)")
        print(f"  Error  : {e}")
    finally:
        if tmp.exists():
            tmp.unlink()

In [ ]:
# ── Run heatmap notebook ──────────────────────────────

if RUN_HEATMAP:
    print(f"\n{'='*55}")
    print(f"  08 · Heatmap Visualization")
    print(f"{'='*55}")

    tmp = pathlib.Path(tempfile.mktemp(suffix=".ipynb"))
    t0 = time.time()
    try:
        pm.execute_notebook(
            "08_heatmap_visualization.ipynb", str(tmp),
            kernel_name=KERNEL_NAME,
            parameters={},
            progress_bar=False,
        )
        elapsed = time.time() - t0
        print(f"  Status : OK  ({elapsed:.1f} s)")
    except pm.exceptions.PapermillExecutionError as e:
        elapsed = time.time() - t0
        print(f"  Status : FAILED  ({elapsed:.1f} s)")
        print(f"  Error  : {e}")
    finally:
        if tmp.exists():
            tmp.unlink()

print(f"\n{'#'*55}")
print(f"  GRID FINDING PIPELINE COMPLETE")
print(f"  Combined CSV: {combined_path}")
print(f"{'#'*55}")